# Common Libraries

In [1]:
import os, sys, shutil
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
from matplotlib.gridspec import GridSpec
from ipywidgets import interact, IntSlider, FloatSlider, Layout, Button, Output, VBox, HBox

if shutil.which("nvidia-smi") is not None:
    os.environ["MUJOCO_GL"] = "egl"
import mujoco

# Custom Libraries

In [2]:
sys.path.append("/home/seojin/Seojin_commonTool/Module")
from biomechanics_util import get_mujoco_joints, get_simulated_img, get_dependent_joints

# Params

In [3]:
model_path = "/mnt/sdb2/DeepProprioception/Projects/DP01_mri/Myosuite/Model/scaled_model_cvt3_mocap.xml"
qpos_data_path = "/mnt/sdb2/DeepProprioception/Projects/DP01_mri/Myosuite/Model/IK_files/trial1_IK_qpos.csv"

target_joint_name = "shoulder_elv"

is_init_byData = True

# Initialize model

In [4]:
model = mujoco.MjModel.from_xml_path(model_path)
mj_data = mujoco.MjData(model)

if is_init_byData:
    qpos_data = pd.read_csv(qpos_data_path, index_col = 0)
    mj_data.qpos = qpos_data.loc[0].values
    mujoco.mj_forward(model, mj_data)
else:
    key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, "default-pose")
    mujoco.mj_resetDataKeyframe(model, mj_data, key_id)
    mujoco.mj_forward(model, mj_data)

# Load data

In [5]:
# Joint
joint_names = [mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, idx) for idx in range(model.njnt)]
dependent_joint_names, independent_joint_names = get_mujoco_joints(model_path)

# Renderer

In [6]:
# Renderer
renderer = mujoco.Renderer(model, height=400, width=600)
scene_option = mujoco.MjvOption()
scene_option.frame = mujoco.mjtFrame.mjFRAME_WORLD

# Camera
camera = mujoco.MjvCamera()
camera.azimuth = 0
camera.elevation = -110
camera.distance = 2.0

# Torque simulation

In [7]:
dof_addrs = {}
for j_name in joint_names:
    jid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, j_name)
    dof_addrs[j_name] = model.jnt_dofadr[jid]

sliders = {}
for j_name in joint_names:
    sliders[j_name] = FloatSlider(
        value=0.0,
        min=-100.0,  # 필요에 따라 범위를 조절하세요
        max=100.0,
        step=1.0,
        description=f'{j_name}:',
        style={'description_width': 'initial'}, # 이름이 잘리지 않도록 설정
        layout=Layout(width="300px")
    )

apply_btn = Button(description='Apply Forces & Step', button_style='success')
reset_btn = Button(description='Reset', button_style='danger')
output_area = Output()

def show_frame():
    with output_area:
        output_area.clear_output(wait=True)
        renderer.update_scene(mj_data, camera=camera, scene_option=scene_option)

        fig, axis = plt.subplots(1, figsize=(8, 6))
        axis.imshow(renderer.render())
        axis.axis("off")
        plt.tight_layout()
        plt.show()

def on_apply_btn_clicked(b):
    for j_name in joint_names:
        addr = dof_addrs[j_name]
        mj_data.qfrc_applied[addr] = sliders[j_name].value
    
    mujoco.mj_step(model, mj_data)
    
    show_frame()

def on_reset_btn_clicked(b):
    for slider in sliders.values():
        slider.value = 0.0
    
    if is_init_byData:
        mj_data.qpos = qpos_data.loc[0].values
    else:
        key_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_KEY, "default-pose")
        mujoco.mj_resetDataKeyframe(model, mj_data, key_id)
    
    mujoco.mj_forward(model, mj_data)
    show_frame()

apply_btn.on_click(on_apply_btn_clicked)
reset_btn.on_click(on_reset_btn_clicked)

slider_list = list(sliders.values())
split_sliders = np.array_split(slider_list, 3)
sliders_box = HBox([
    VBox(list(split_sliders[0])), 
    VBox(list(split_sliders[1])), 
    VBox(list(split_sliders[2]))
])
buttons_box = HBox([apply_btn, reset_btn])
ui_panel = VBox([sliders_box, buttons_box])

display(ui_panel, output_area)

show_frame()

Output()